# LULCC–GDSTEM community area fractions

This notebook calculates grid-cell area fractions for TEM communities 1–23 over the selected years. It:

1. reads cohort-level LULCC data in chunks;
2. selects one vegetation code according to management state;
3. audits the classification of every distinct code combination;
4. calculates annual community fractions and their arithmetic multi-year means;
5. reports coverage for all 23 communities; and
6. optionally checks coordinate coverage in an existing Stage 2 GFED summary.

The calculations, class mappings, file naming, and output columns follow the supplied `01_lulcc_community.py`. English comments and step descriptions explain the processing, and selected compact Python expressions have been expanded. The notebook structure follows the local `fluxnet-gdstem-site-selection.ipynb` in the repository linked by the user. The remote copy could not be retrieved for comparison.

This is a separate review copy. The original script is unchanged. Full server input processing and server ECD verification have not been performed.

## 1. Imports and configuration

Edit the input and output paths before running. The year interval includes both endpoints. Keep `REUSE_COMPLETED_DATABASE=False` for the first run. Set it to `True` only when the same input and year range have already been scanned completely. A completed scan can be reused after classification rules are reviewed. `FAIL_ON_UNCLASSIFIED=True` writes the classification audits and stops before exporting final fractions if valid area remains unclassified.

In [ ]:
"""Stage 1: calculate TEM community fractions for communities 1-23.

Each valid cohort contributes its area to one management-state-selected category:
agstate 0 uses subtype; agstate 1-6 uses currentveg; agstate 7 is community 15.
Unclassified area remains in the annual denominator. Multi-year fractions are
arithmetic means of annual fractions. A missing community receives zero in an
observed grid-year; a year with no records for the grid is not added as zero.

The workflow reads large CSV files in chunks and stores intermediate sums in
SQLite on disk. It retains the original 11 fraction-table columns, mappings,
filenames, metadata keys, and audit labels. It does not import config.py or edit
the input LULCC file, model ECD files, or earlier Stage 1 results.

Provenance reported by the supplied script (not reverified in this rewrite):
  GDSTEM-FIRE/lulcdat_gds.cpp: 15-column LULCC layout;
  xtem_gds_fire_gofile_emit.cpp:3633-3653: cmnt/agcmnt assignment;
  ttem_gds_fire_emit.cpp:1162-1174,4188: management state and impervious area;
  dat_ecd_files/tcomm45_lulc.ecd: vegetation-to-community mapping.
The server ECD and full server input have not been validated here."""

import csv
import json
import sqlite3
import time
from pathlib import Path

import numpy as np
import pandas as pd


# Configuration: review these settings before running.

# Set this to the LULCC input file verified for your server.
LULCC_FILE = Path('/quobyte/emoniergrp/TRENDY/input-files/global-input/gcb2025/lulcc-TRENDY-1700-2024.csv')

# Use the Stage 1 / Stage 2 output directory with the new output filenames.
# The full_1_23 suffix keeps the earlier fraction table and SQLite file separate.
OUTPUT_DIR = Path('/quobyte/emoniergrp/luyy123/calibration_gridcell_selection_2002_2022_v2')
WORK_DIR = OUTPUT_DIR  # Optional: use a scratch directory with enough disk space and write permission.

# Keep this inclusive year range consistent with Stage 2.
START_YEAR = 2002
END_YEAR = 2022
LULCC_CHUNKSIZE = 300_000  # Reduce to 50_000 if memory is limited.

# Keep False for the first scan. An existing database causes the run to stop.
# Set True only after this version has finished scanning the entire input CSV.
# Reuse supports reclassification; the earlier 01_work_lulcc.sqlite is incompatible.
REUSE_COMPLETED_DATABASE = False

# Keep True to write audits and stop when valid area cannot be classified.
# The completed database is retained and can be reused after reviewing the mapping.
# False allows partial classification: unknown area stays in each denominator.
FAIL_ON_UNCLASSIFIED = True

## 2. Community mappings and input conventions

Each valid cohort contributes to exactly one selected community:

| Management state | Selected classification |
| --- | --- |
| `agstate = 0` | Map `subtype` |
| `agstate = 1–6` | Map `currentveg`, including the explicit override |
| `agstate = 7` | Community 15, impervious surfaces |
| Other or missing state | Unclassified, represented by 0 |

The supplied mapping is retained, including `currentveg 58 → community 21` in managed states. Codes 50–53 are generic crop parameter proxies, not observed crop-species identifications. Mixed codes must already be split into atomic cohorts. Reserved codes are not counted as ice. The ECD provenance below is carried over from the supplied script and has not been independently checked against the server.

In [ ]:
# All 23 supported community names; some may be absent from the input data.
COMMUNITY_NAMES = {
    1: 'Ice', 2: 'Alpine Tundra and Polar Desert', 3: 'Moist and Wet Tundra',
    4: 'Boreal Forests', 5: 'Temperate Coniferous Forests',
    6: 'Temperate Deciduous Forests', 7: 'Temperate Broadleaved Evergreen Forest',
    8: 'Short Grassland', 9: 'Tall Grassland', 10: 'Mediterranean Shrubland',
    11: 'Xeric Shrubland', 12: 'Xeric Woodland',
    13: 'Tropical Evergreen Forests', 14: 'Tropical Deciduous Forests',
    15: 'Impervious Surfaces', 16: 'Turf Lawn', 17: 'Soybean', 18: 'Maize',
    19: 'Fruit tree', 20: 'Switchgrass', 21: 'Vegetable Farm',
    22: 'Pasture', 23: 'Rangeland',
}

# Input vegetation code -> community number. Verify the ECD version separately.
# Dictionary keys are input vegetation codes; values are community numbers 1-23.
# The mapping includes natural vegetation, impervious surfaces, lawn, and crops.
# Codes 50-53 are generic C3/C4 annual/perennial crop parameter proxies.
# For example, 50 -> 17 does not mean all observed area is soybean; the audit flags this.
MODEL_ATOMIC_VEG_TO_COMMUNITY = {
    1: 1, 2: 2, 3: 3, 4: 4, 6: 4, 7: 3,
    8: 5, 9: 6, 11: 7, 13: 6, 14: 9, 15: 6, 16: 9,
    17: 9, 18: 9, 19: 8, 20: 9, 21: 10, 22: 11, 23: 12,
    24: 13, 25: 14, 28: 13, 29: 9, 30: 14, 31: 9, 32: 13, 33: 11,
    47: 15, 48: 16,
    50: 17, 51: 18, 52: 19, 53: 20,
    54: 17, 55: 18, 56: 19, 57: 20,
    58: 18,  # Original local tcomm mapping; the explicit currentveg override is defined below.
    60: 22, 61: 23,
}

# Map currentveg 58 to community 21 only when agstate is 1-6.
# The original script reports local tcomm code 58 as Vegetable Farm with subtype1=18.
# This area classification uses community 21, explicitly differing from that parameter mapping.
# If the server also parameterizes 58 as 18, this table and model parameters still differ.
CURRENTVEG_CLASS_OVERRIDES = {58: 21}

# Mixed vegetation must already be split into atomic cohorts in the input.
# Codes 40-46 are mixed urban types and also require review if selected for classification.
NONATOMIC_VEG_CODES = {5, 10, 12, 26, 27, 40, 41, 42, 43, 44, 45, 46}

# Reserved local tcomm codes must not be interpreted as real ice area.
RESERVED_VEG_CODES = {34, 35, 36, 37, 38, 39, 49, 59}

# Optional: check whether an existing Stage 2 summary covers the target coordinates.
# This reads only summary coordinates; it does not recompute or audit raw GFED data.
GFED_SUMMARY_FOR_COVERAGE = OUTPUT_DIR / '02_gfed_by_class_summary_2002_2022.csv'

# Processing constants and accepted input-column names.

SCALE = 1_000_000
MISSING_CODE = -999
SCHEMA_VERSION = 'stage1_full_1_23_v1'
KEYS = ['lon_i', 'lat_i', 'year', 'subtype', 'currentveg', 'agstate']
ALIASES = {
    'lon': {'lon', 'longitude'}, 'lat': {'lat', 'latitude'},
    'year': {'year', 'yr', 'iyear'},
    'cohort_area': {'cohort_area', 'chrtarea', 'cohortarea'},
    'subtype': {'subtype', 'sub_type', 'isubtype'},
    'currentveg': {'currentveg', 'current_veg', 'current_vegetation'},
    'agstate': {'agstate', 'ag_state', 'lc_state', 'land_use_state'},
}

## 3. Output paths and input columns

`output_paths()` names the output files; `input_columns()` detects column aliases or uses the original headerless column positions. The headerless input requires at least 12 columns. Column positions start at zero. The explicit loops below replace compact comprehensions so that header detection is easier to follow.

In [ ]:
def output_paths():
    """Build filenames using the selected year range and the full_1_23 suffix."""
    tag = f'{START_YEAR}_{END_YEAR}_full_1_23'
    paths = {
        'fractions': OUTPUT_DIR / f'01_community_fraction_{tag}.csv',
        'audit': OUTPUT_DIR / f'01_classification_audit_{tag}.csv',
        'unclassified': OUTPUT_DIR / f'01_unclassified_{tag}.csv',
        'coverage': OUTPUT_DIR / f'01_class_coverage_{tag}.csv',
        'grid': OUTPUT_DIR / f'01_grid_coordinates_{tag}.csv',
        'stage2_missing_grids': OUTPUT_DIR / f'01_stage2_missing_grids_{tag}.csv',
        'manifest': OUTPUT_DIR / f'01_run_summary_{tag}.json',
        'db': WORK_DIR / f'01_work_lulcc_{tag}.sqlite',
    }
    return paths


def input_columns():
    """Detect a header or use the original headerless LULCCHRT column positions.
    
    Return the pandas header setting, columns to read, and canonical column names."""
    with LULCC_FILE.open(encoding='utf-8-sig', newline='') as fh:
        row = next(csv.reader(fh, skipinitialspace=True), None)
    if not row:
        raise ValueError('The LULCC input file is empty.')
    normalized = []
    for column in row:
        normalized.append(column.strip().lower().replace(' ', '_'))
    has_header = False
    for column in normalized:
        if column in ALIASES['lon']:
            has_header = True
            break

    if has_header:
        rename = {}
        for field, alternatives in ALIASES.items():
            found = []
            for position, column in enumerate(normalized):
                if column in alternatives:
                    found.append(row[position])
            if len(found) != 1:
                raise ValueError(f'Cannot uniquely identify field {field}; header={row}')
            rename[found[0]] = field
        return 0, list(rename), rename
    if len(row) < 12:
        raise ValueError(f'Headerless input requires at least 12 columns; found {len(row)}.')
    # Zero-based columns: lon=0, lat=1, year=3, area=7, currentveg=9, subtype=10, agstate=11.
    positions = {0: 'lon', 1: 'lat', 3: 'year', 7: 'cohort_area',
                 9: 'currentveg', 10: 'subtype', 11: 'agstate'}
    return None, list(positions), positions

## 4. Assign one community to each code combination

`classify_record()` returns the community number, the branch used, the selected code, and an audit note. It checks only the vegetation code used by the current management-state branch. For managed states, `copy()` and `update()` apply overrides while keeping the base dictionary unchanged.

In [ ]:
def classify_record(subtype, currentveg, agstate):
    """Assign one community to a cohort using its management state.
    
    Return (community, branch, selected_code, note). Community 0 means unclassified,
    not ice. Validate only the vegetation code used by the selected state branch.
    These are area-classification rules, not a simulation of TEM vegetation dynamics."""
    if agstate == 7:
        return 15, 'impervious_by_agstate', currentveg, 'agstate_7'
    if agstate == 0:
        code = subtype
        branch = 'subtype_by_agstate_0'
        mapping = MODEL_ATOMIC_VEG_TO_COMMUNITY
    elif agstate in (1, 2, 3, 4, 5, 6):
        code = currentveg
        branch = 'currentveg_by_agstate_1_6'
        # copy() keeps the base mapping unchanged; update() applies the overrides.
        mapping = MODEL_ATOMIC_VEG_TO_COMMUNITY.copy()
        mapping.update(CURRENTVEG_CLASS_OVERRIDES)
    else:
        return 0, 'unclassified', agstate, 'missing_or_unsupported_agstate'
    if code == MISSING_CODE:
        return 0, branch, code, 'missing_selected_vegetation_code'
    if code in NONATOMIC_VEG_CODES:
        return 0, branch, code, 'non_atomic_selected_code_requires_split_cohorts'
    if code in RESERVED_VEG_CODES:
        return 0, branch, code, 'reserved_code_not_real_ice'
    community = mapping.get(code)
    if community not in COMMUNITY_NAMES:
        return 0, branch, code, 'unmapped_selected_vegetation_code'
    note = 'mapped'
    if agstate in (1, 2, 3, 4, 5, 6) and code in CURRENTVEG_CLASS_OVERRIDES:
        note = 'explicit_user_numbering_override_from_model_mapping'
    elif code in (50, 51, 52, 53):
        note = 'generic_crop_parameter_proxy_not_species_observation'
    elif agstate == 0 and currentveg in range(47, 62):
        # State 0 uses subtype even if currentveg retains an old managed-vegetation code.
        note = 'state_0_uses_subtype_currentveg_not_used'
    return community, branch, code, note

## 5. Create or reuse the disk database

SQLite stores intermediate data on disk so the full CSV does not have to fit in memory. The database records its schema, input path, file size, modification time, selected years, and whether scanning completed. Reuse requires these values to match. The source signature is a metadata check, not a content checksum. `meta_set()` and `meta_get()` save and retrieve JSON metadata.

In [ ]:
def source_signature():
    """Record input identity and years to validate reuse of a completed database."""
    stat = LULCC_FILE.stat()
    return {'path': str(LULCC_FILE.resolve()), 'bytes': stat.st_size,
            'mtime_ns': stat.st_mtime_ns, 'start_year': START_YEAR, 'end_year': END_YEAR}


def meta_set(conn, key, value):
    """Save one metadata value as JSON text in the database."""
    json_value = json.dumps(value, ensure_ascii=False)
    conn.execute('INSERT OR REPLACE INTO metadata VALUES (?,?)', (key, json_value))


def meta_get(conn, key):
    """Read one metadata value; return None if the key is absent."""
    row = conn.execute('SELECT value FROM metadata WHERE key=?', (key,)).fetchone()
    if row is None:
        return None
    return json.loads(row[0])


def connect_db(path, reuse):
    """Create or validate a disk database containing raw code combinations.
    
    The stored combinations allow reclassification without rescanning the CSV."""
    conn = sqlite3.connect(path)
    # SQLite settings: keep temporary tables on disk and use about 32 MB of cache.
    # Retain these settings when the input is too large to hold in memory.
    conn.execute('PRAGMA journal_mode=DELETE')
    conn.execute('PRAGMA synchronous=NORMAL')
    conn.execute('PRAGMA temp_store=FILE')
    conn.execute('PRAGMA cache_size=-32000')
    if reuse:
        try:
            ok = (meta_get(conn, 'schema') == SCHEMA_VERSION and
                  meta_get(conn, 'scan_complete') is True and
                  meta_get(conn, 'source') == source_signature())
        except sqlite3.Error:
            ok = False
        if not ok:
            conn.close()
            raise ValueError('Cannot reuse database: schema, completed-scan flag, input file, or year range does not match.')
        return conn
    conn.executescript('''
        CREATE TABLE metadata (key TEXT PRIMARY KEY, value TEXT NOT NULL);
        CREATE TABLE cohort_year (
          lon_i INTEGER NOT NULL, lat_i INTEGER NOT NULL, year INTEGER NOT NULL,
          subtype INTEGER NOT NULL, currentveg INTEGER NOT NULL, agstate INTEGER NOT NULL,
          area REAL NOT NULL, n_rows INTEGER NOT NULL,
          PRIMARY KEY(lon_i,lat_i,year,subtype,currentveg,agstate)
        ) WITHOUT ROWID;
    ''')
    meta_set(conn, 'schema', SCHEMA_VERSION)
    meta_set(conn, 'scan_complete', False)
    meta_set(conn, 'source', source_signature())
    conn.commit()
    return conn

## 6. Read and accumulate the LULCC data

Read the CSV in chunks, select the requested years, and retain rows with finite coordinates, valid latitude, and finite positive cohort area. Missing classification codes remain in the data with a sentinel value so their area can remain in the denominator.

Longitude is normalized to `[-180, 180)`. Coordinates are rounded to integer keys at a scale of 1,000,000. Within each chunk, rows with the same grid, year, subtype, current vegetation, and state are summed. SQLite then adds matching rows across chunks. `n_rows` retains the number of original cohort records represented by each sum. The SQL insert/update operation is retained because it supports accumulation on disk.

In [ ]:
def ingest(conn):
    """Scan the CSV in chunks and store selected years with valid positive areas.
    
    Missing classification codes do not remove otherwise valid area."""
    header, usecols, rename = input_columns()
    print('Detected', 'headered' if header == 0 else 'headerless',
          'LULCC; column map:', rename, flush=True)
    reader = pd.read_csv(LULCC_FILE, header=header, usecols=usecols,
                         chunksize=LULCC_CHUNKSIZE, skipinitialspace=True,
                         low_memory=False, encoding='utf-8-sig', on_bad_lines='error')
    # Repeated grid/year/code keys add their area and row count to the stored sums.
    query = '''INSERT INTO cohort_year VALUES (?,?,?,?,?,?,?,?)
               ON CONFLICT(lon_i,lat_i,year,subtype,currentveg,agstate)
               DO UPDATE SET area=area+excluded.area, n_rows=n_rows+excluded.n_rows'''
    raw_rows = kept_rows = 0
    start = time.monotonic()
    for i, chunk in enumerate(reader, 1):
        chunk = chunk.rename(columns=rename)
        raw_rows += len(chunk)
        for field in ALIASES:
            chunk[field] = pd.to_numeric(chunk[field], errors='coerce')
        # Retain valid positive areas even when classification fields are missing.
        keep = (chunk.year.between(START_YEAR, END_YEAR) &
                np.isfinite(chunk.lon) & np.isfinite(chunk.lat) &
                chunk.lat.between(-90, 90) & np.isfinite(chunk.cohort_area) &
                (chunk.cohort_area > 0))
        chunk = chunk.loc[keep].copy()
        if chunk.empty:
            if i % 100 == 0:
                print(f'chunks={i}, raw={raw_rows:,}, retained={kept_rows:,}', flush=True)
            continue
        if not chunk.year.eq(np.floor(chunk.year)).all():
            raise ValueError('The selected records contain noninteger years.')
        for field in ('subtype', 'currentveg', 'agstate'):
            finite = np.isfinite(chunk[field])
            bad = finite & chunk[field].ne(np.floor(chunk[field]))
            if bad.any():
                raise ValueError(f'{field} contains noninteger codes: {chunk.loc[bad, field].head().tolist()}')
            # Store missing/nonfinite codes with a sentinel; classification checks the selected branch.
            chunk.loc[~finite, field] = MISSING_CODE
            # Check the range before converting to int32 to avoid integer overflow.
            if ((chunk[field] < np.iinfo(np.int32).min) |
                    (chunk[field] > np.iinfo(np.int32).max)).any():
                raise ValueError(f'{field} contains codes outside the int32 range.')
            chunk[field] = chunk[field].astype('int32')
        lon = ((chunk.lon.to_numpy() + 180.0) % 360.0) - 180.0
        chunk['lon_i'] = np.rint(lon * SCALE).astype('int32')
        chunk['lat_i'] = np.rint(chunk.lat.to_numpy() * SCALE).astype('int32')
        chunk['year'] = chunk.year.astype('int32')
        # Sum within each chunk, then across chunks in SQLite; retain original cohort row counts.
        grouped = chunk.groupby(KEYS, as_index=False, sort=False).agg(
            area=('cohort_area', 'sum'), n_rows=('cohort_area', 'size'))
        rows = grouped[KEYS + ['area', 'n_rows']].itertuples(index=False, name=None)
        with conn:
            conn.executemany(query, rows)
        kept_rows += len(chunk)
        if i % 20 == 0:
            print(f'chunks={i}, raw={raw_rows:,}, {START_YEAR}-{END_YEAR}={kept_rows:,}, '
                  f'elapsed={time.monotonic()-start:.0f}s', flush=True)
    if not kept_rows:
        raise ValueError('No records passed the year and area filters.')
    meta_set(conn, 'raw_rows', raw_rows)
    meta_set(conn, 'kept_rows', kept_rows)
    meta_set(conn, 'scan_complete', True)
    conn.commit()
    print('Completed LULCC scan, retained rows:', f'{kept_rows:,}', flush=True)

## 7. Export classification audits

Classify only distinct `(subtype, currentveg, agstate)` combinations and join the results to the accumulated area. Explicit loops replace nested list construction and tuple unpacking. Export both the full audit and the unclassified combinations, including empty-table headers.

Audit area is a sum across all included grids and years. It is not land area for a single year.

In [ ]:
def export_query(conn, query, path, names=False):
    """Write SQL query results to CSV in chunks, including an empty-file header.
    
    Set names=True to add the community_name column."""
    first = True
    for part in pd.read_sql_query(query, conn, chunksize=75_000):
        if names:
            part['community_name'] = part.community_type.map(COMMUNITY_NAMES)
        part.to_csv(path, index=False, mode='w' if first else 'a', header=first)
        first = False
    if first:
        # If pandas yields no chunks, use LIMIT 0 to obtain and write the column names.
        pd.read_sql_query('SELECT * FROM (' + query.rstrip(';') + ') LIMIT 0', conn).to_csv(path, index=False)


def build_classification(conn, paths):
    """Classify distinct code combinations and export full and unclassified audits."""
    conn.execute('DROP TABLE IF EXISTS mapping')
    conn.execute('''CREATE TABLE mapping (
        subtype INTEGER, currentveg INTEGER, agstate INTEGER,
        community_type INTEGER, classification_branch TEXT,
        selected_vegetation_code INTEGER, classification_note TEXT,
        PRIMARY KEY(subtype,currentveg,agstate)) WITHOUT ROWID''')
    cursor = conn.execute('SELECT DISTINCT subtype,currentveg,agstate FROM cohort_year')
    while True:
        batch = cursor.fetchmany(500)
        if not batch:
            break
        mapping_rows = []
        for subtype, currentveg, agstate in batch:
            community, branch, selected_code, note = classify_record(
                subtype, currentveg, agstate
            )
            mapping_rows.append((
                subtype, currentveg, agstate, community, branch, selected_code, note
            ))
        conn.executemany('INSERT INTO mapping VALUES (?,?,?,?,?,?,?)', mapping_rows)
    conn.commit()
    # One audit row per raw code combination; area is summed across all grids and years.
    audit_query = '''SELECT c.subtype,c.currentveg,c.agstate,m.community_type,
        m.classification_branch,m.selected_vegetation_code,m.classification_note,
        SUM(c.n_rows) AS n_cohort_rows, COUNT(*) AS n_grid_year_code_combinations,
        SUM(c.area) AS total_cohort_area_across_years
        FROM cohort_year c JOIN mapping m
        USING(subtype,currentveg,agstate)
        GROUP BY c.subtype,c.currentveg,c.agstate
        ORDER BY m.community_type,c.agstate,c.currentveg,c.subtype'''
    export_query(conn, audit_query, paths['audit'], names=True)
    export_query(conn, 'SELECT * FROM (' + audit_query + ') WHERE community_type=0',
                 paths['unclassified'], names=True)
    unknown_rows, unknown_area = conn.execute('''
        SELECT COALESCE(SUM(c.n_rows),0),COALESCE(SUM(c.area),0)
        FROM cohort_year c JOIN mapping m USING(subtype,currentveg,agstate)
        WHERE m.community_type=0''').fetchone()
    print('Unclassified:', f'{unknown_rows:,}', 'cohort rows; summed area=', unknown_area,
          '; audit:', paths['unclassified'], flush=True)
    return int(unknown_rows), float(unknown_area)

## 8. Calculate annual and multi-year fractions

For grid $g$, year $y$, and community $c$:

$$A_{g,y}=\sum_{\text{all retained cohorts}} a_i,\qquad
A_{g,y,c}=\sum_{\text{cohorts assigned to }c}a_i,$$

$$f_{g,y,c}=\frac{A_{g,y,c}}{A_{g,y}},\qquad
\bar f_{g,c}=\frac{\sum_{y\in Y_g}f_{g,y,c}}{|Y_g|}.$$

$Y_g$ contains only years with retained records for the grid. A community absent in one of those years contributes zero to the mean and makes the minimum zero. A year with no grid records is excluded. Unclassified area stays in $A_{g,y}$, so known-community fractions can sum to less than one if partial classification is allowed.

`mean_present_community_area` averages only years when that community is present. `mean_total_cohort_area` averages all observed grid-years. The main fraction table includes only grid/community pairs that occur at least once; the separate coverage table always lists communities 1–23.

The SQL steps build annual totals, build community areas, check conservation for every grid-year, and calculate the statistics. These steps remain on disk to preserve the original large-file workflow.

In [ ]:
def aggregate(conn, paths):
    """Calculate annual fractions and multi-year statistics with the original formulas."""
    conn.executescript('''
        DROP TABLE IF EXISTS grid_year;
        DROP TABLE IF EXISTS community_year;
        CREATE TABLE grid_year AS
          SELECT lon_i,lat_i,year,SUM(area) AS total_area
          FROM cohort_year GROUP BY lon_i,lat_i,year;
        CREATE UNIQUE INDEX idx_grid_year ON grid_year(lon_i,lat_i,year);
        CREATE TABLE community_year AS
          SELECT c.lon_i,c.lat_i,c.year,m.community_type,SUM(c.area) AS community_area
          FROM cohort_year c JOIN mapping m USING(subtype,currentveg,agstate)
          WHERE m.community_type BETWEEN 1 AND 23
          GROUP BY c.lon_i,c.lat_i,c.year,m.community_type;
        CREATE INDEX idx_community_year ON community_year(lon_i,lat_i,year);
    ''')
    # Check area conservation separately for every grid-year.
    # A global sum alone could hide discrepancies between individual grids or years.
    violations = conn.execute('''
        WITH classified AS (
          SELECT lon_i,lat_i,year,SUM(community_area) AS area
          FROM community_year GROUP BY lon_i,lat_i,year),
        unknown AS (
          SELECT c.lon_i,c.lat_i,c.year,SUM(c.area) AS area
          FROM cohort_year c JOIN mapping m USING(subtype,currentveg,agstate)
          WHERE m.community_type=0 GROUP BY c.lon_i,c.lat_i,c.year)
        SELECT COUNT(*) FROM grid_year g
        LEFT JOIN classified a USING(lon_i,lat_i,year)
        LEFT JOIN unknown u USING(lon_i,lat_i,year)
        WHERE ABS(g.total_area-COALESCE(a.area,0)-COALESCE(u.area,0))>
              1e-8+1e-10*ABS(g.total_area)''').fetchone()[0]
    if violations:
        raise ValueError(f'Area conservation failed for {violations} grid-years; export stopped.')
    # Annual denominators contain all valid cohort area, including unclassified area.
    # Community absence counts as zero; years with no grid records are excluded.
    # SQL steps: annual fractions -> community statistics -> observed-year counts.
    # The final mean is sum(annual fraction) / n_years_grid, not a ratio of area sums.
    query = '''WITH yearly_fraction AS (
        SELECT c.lon_i,c.lat_i,c.year,c.community_type,c.community_area,
          c.community_area/g.total_area AS fraction
        FROM community_year c JOIN grid_year g USING(lon_i,lat_i,year)
        WHERE g.total_area>0),
      group_stats AS (
        SELECT lon_i,lat_i,community_type,SUM(fraction) AS sum_fraction,
          MIN(fraction) AS min_present_fraction,MAX(fraction) AS max_fraction,
          COUNT(*) AS n_years_community,AVG(community_area) AS mean_present_community_area
        FROM yearly_fraction GROUP BY lon_i,lat_i,community_type),
      grid_stats AS (
        SELECT lon_i,lat_i,COUNT(*) AS n_years_grid,AVG(total_area) AS mean_total_cohort_area
        FROM grid_year WHERE total_area>0 GROUP BY lon_i,lat_i)
      SELECT g.lon_i,g.lat_i,g.community_type,
        g.sum_fraction/s.n_years_grid AS mean_community_fraction,
        CASE WHEN g.n_years_community<s.n_years_grid THEN 0
             ELSE g.min_present_fraction END AS min_community_fraction,
        g.max_fraction AS max_community_fraction,g.n_years_community,s.n_years_grid,
        g.mean_present_community_area,s.mean_total_cohort_area
      FROM group_stats g JOIN grid_stats s USING(lon_i,lat_i)
      ORDER BY g.community_type,g.lon_i,g.lat_i'''
    columns = ['lon', 'lat', 'community_type', 'community_name',
               'mean_community_fraction', 'min_community_fraction', 'max_community_fraction',
               'n_years_community', 'n_years_grid', 'mean_present_community_area',
               'mean_total_cohort_area']
    first = True
    for part in pd.read_sql_query(query, conn, chunksize=75_000):
        part['lon'] = part.pop('lon_i') / SCALE
        part['lat'] = part.pop('lat_i') / SCALE
        part['community_name'] = part.community_type.map(COMMUNITY_NAMES)
        part[columns].to_csv(paths['fractions'], index=False,
                             mode='w' if first else 'a', header=first)
        first = False
    if first:
        pd.DataFrame(columns=columns).to_csv(paths['fractions'], index=False)
        raise ValueError('No communities could be classified; review the classification audit.')
    # Always list communities 1-23; absent communities have zero observed coverage.
    coverage = pd.DataFrame({'community_type': range(1, 24)})
    stats = pd.read_sql_query('''SELECT community_type,COUNT(*) AS n_grid_years,
        SUM(community_area) AS total_cohort_area_across_years,
        COUNT(DISTINCT printf('%d,%d',lon_i,lat_i)) AS n_grids
        FROM community_year GROUP BY community_type''', conn)
    coverage = coverage.merge(stats, on='community_type', how='left').fillna(0)
    coverage['n_grid_years'] = coverage.n_grid_years.astype('int64')
    coverage['n_grids'] = coverage.n_grids.astype('int64')
    coverage['community_name'] = coverage.community_type.map(COMMUNITY_NAMES)
    coverage = coverage[['community_type', 'community_name', 'n_grids', 'n_grid_years',
                         'total_cohort_area_across_years']]
    coverage.to_csv(paths['coverage'], index=False)
    # Export all target coordinates for comparison with the earlier Stage 2 summary.
    export_query(conn, f'''SELECT lon_i*1.0/{SCALE} AS lon,lat_i*1.0/{SCALE} AS lat,
                 COUNT(*) AS n_years_grid FROM grid_year
                 GROUP BY lon_i,lat_i ORDER BY lat_i,lon_i''', paths['grid'])
    print(coverage.to_string(index=False), flush=True)

## 9. Check Stage 2 coordinate coverage

If the existing Stage 2 summary is available, compare its normalized coordinates with the Stage 1 target grids and export missing coordinates. If the file is absent, record that the check was skipped. This checks coordinate coverage only; it does not validate GFED units, year alignment, category definitions, or raw data quality.

In [ ]:
def check_stage2_grid_coverage(conn, paths):
    """Check target-coordinate coverage in an existing Stage 2 summary.
    
    This does not recompute GFED or check its units, years, categories, or data quality."""
    if not GFED_SUMMARY_FOR_COVERAGE.is_file():
        return {'checked': False, 'reason': 'Stage 2 summary not found',
                'path': str(GFED_SUMMARY_FOR_COVERAGE)}
    conn.execute('DROP TABLE IF EXISTS temp.old_gfed_grid')
    conn.execute('''CREATE TEMP TABLE old_gfed_grid (lon_i INTEGER,lat_i INTEGER,
                    PRIMARY KEY(lon_i,lat_i)) WITHOUT ROWID''')
    for part in pd.read_csv(GFED_SUMMARY_FOR_COVERAGE, usecols=['lon', 'lat'], chunksize=150_000):
        lon = ((part.lon.to_numpy() + 180.0) % 360.0) - 180.0
        if not (np.isfinite(lon).all() and np.isfinite(part.lat).all()):
            raise ValueError('Existing Stage 2 coordinates contain nonfinite values; coverage cannot be confirmed.')
        pairs = pd.DataFrame({'lon_i': np.rint(lon*SCALE).astype('int64'),
                              'lat_i': np.rint(part.lat.to_numpy()*SCALE).astype('int64')}).drop_duplicates()
        conn.executemany('INSERT OR IGNORE INTO old_gfed_grid VALUES (?,?)',
                         pairs.itertuples(index=False, name=None))
    missing_query = f'''SELECT g.lon_i*1.0/{SCALE} AS lon,g.lat_i*1.0/{SCALE} AS lat
        FROM (SELECT DISTINCT lon_i,lat_i FROM grid_year) g
        LEFT JOIN old_gfed_grid o USING(lon_i,lat_i) WHERE o.lon_i IS NULL
        ORDER BY g.lat_i,g.lon_i'''
    missing = conn.execute('SELECT COUNT(*) FROM (' + missing_query + ')').fetchone()[0]
    export_query(conn, missing_query, paths['stage2_missing_grids'])
    return {'checked': True, 'missing_target_grids': int(missing),
            'all_new_target_coordinates_already_in_stage2': missing == 0,
            'scope': 'coordinate coverage only; not a fresh unit/year/category/data-quality audit'}

## 10. Run the workflow

Run the preceding cells in order, then run this cell. This is the only cell that starts the input scan and writes processing outputs. `main()` validates configuration, opens the database, scans or reuses it, writes classification audits, computes fractions, and checks Stage 2 coordinate coverage. The run-summary JSON records completion or failure. On a classification stop, review the unclassified audit before changing the mapping and reusing the completed database.

In [ ]:
def main():
    """Run scanning, classification, aggregation, and the optional coverage check."""
    if START_YEAR > END_YEAR or LULCC_CHUNKSIZE <= 0:
        raise ValueError('Invalid year range or CSV chunk size.')
    if set(COMMUNITY_NAMES) != set(range(1, 24)):
        raise ValueError('COMMUNITY_NAMES must contain exactly community numbers 1-23.')
    if not LULCC_FILE.is_file():
        raise FileNotFoundError(LULCC_FILE)
    paths = output_paths()
    exists = paths['db'].exists()
    if exists and not REUSE_COMPLETED_DATABASE:
        raise FileExistsError(f"The full_1_23 database already exists: {paths['db']}; do not delete it automatically.\n"
                              'For a completed scan that needs reclassification, set REUSE_COMPLETED_DATABASE=True.')
    if REUSE_COMPLETED_DATABASE and not exists:
        raise FileNotFoundError(f"No full_1_23 database is available for reuse: {paths['db']}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    conn = connect_db(paths['db'], reuse=exists)
    manifest = {'status': 'running', 'source': source_signature(),
                'classification_definition': 'state0:subtype; state1-6:currentveg; state7:community15',
                'currentveg_class_overrides': CURRENTVEG_CLASS_OVERRIDES,
                'paths': {}}
    for name, path in paths.items():
        manifest['paths'][name] = str(path)
    try:
        if not exists:
            ingest(conn)
        else:
            print('Reusing fully scanned NEW database; CSV scan skipped.', flush=True)
        unknown_rows, unknown_area = build_classification(conn, paths)
        manifest.update(unclassified_cohort_rows=unknown_rows,
                        unclassified_total_cohort_area_across_years=unknown_area)
        if unknown_rows and FAIL_ON_UNCLASSIFIED:
            manifest['status'] = 'needs_classification_review'
            raise ValueError(f"Unclassified area was found. Review {paths['unclassified']}.\n"
                             'The completed database has been retained. Review the rules, then reuse it to repeat classification and aggregation.')
        aggregate(conn, paths)
        manifest['stage2_grid_coverage'] = check_stage2_grid_coverage(conn, paths)
        if unknown_rows == 0:
            manifest['status'] = 'complete'
        else:
            manifest['status'] = 'complete_with_unclassified_area' 
        print('Stage 2 coordinate coverage:', manifest['stage2_grid_coverage'], flush=True)
        print('Saved NEW community fractions:', paths['fractions'], flush=True)
    except Exception as exc:
        if manifest['status'] == 'running':
            manifest['status'] = 'failed'
        manifest['error'] = str(exc)
        raise
    finally:
        conn.close()
        paths['manifest'].write_text(json.dumps(manifest, ensure_ascii=False, indent=2),
                                     encoding='utf-8')
        print('Run summary:', paths['manifest'], flush=True)


main()

## Pipeline outputs

For the default years, the filename tag is `2002_2022_full_1_23`.

| Output | Contents |
| --- | --- |
| `01_community_fraction_<tag>.csv` | Original 11 columns: coordinates, community identity, fraction statistics, year counts, and area statistics |
| `01_classification_audit_<tag>.csv` | Every distinct raw classification combination and its assigned class |
| `01_unclassified_<tag>.csv` | Combinations requiring classification review |
| `01_class_coverage_<tag>.csv` | Coverage for communities 1–23, including absent classes |
| `01_grid_coordinates_<tag>.csv` | All retained target grids and observed year counts |
| `01_stage2_missing_grids_<tag>.csv` | Target coordinates missing from the existing Stage 2 summary, when checked |
| `01_run_summary_<tag>.json` | Input identity, output paths, classification totals, coverage check, and run status |
| `01_work_lulcc_<tag>.sqlite` | Disk-based intermediate sums and completed-scan metadata |

The SQLite database is stored under `WORK_DIR`; exported tables and the run summary are stored under `OUTPUT_DIR`. With the default stop-on-unclassified setting, classification audits and the run summary are written before the stop; final fractions are generated after classification review succeeds. Output names and database reuse behavior match the supplied script, so use a separate output directory if you want to compare both versions without replacing generated tables.